In [11]:
# --- 필요한 라이브러리 불러오기 ---
import os
import json
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [12]:
# ==============================================================================
# ✨ 핵심 기능 함수: 예측 수행 ✨
# ==============================================================================
def predict_phishing(model, tokenizer, sentences):
    """
    [함수 기능]
    이 함수는 모델, 토크나이저, 그리고 문장 리스트를 입력받아,
    각 문장이 보이스피싱인지 아닌지 예측하고 그 결과를 반환합니다.
    
    [사용 목적]
    학습 전/후 모델에 동일한 예측 작업을 반복해야 하므로,
    코드를 중복해서 쓰지 않고 이 함수를 재사용하기 위해 만들었습니다.
    유지보수가 쉬워지는 가장 기본적인 방법입니다.
    """
    
    # 예측 결과를 저장할 빈 리스트
    results = []
    
    # 입력된 모든 문장에 대해 하나씩 예측을 수행
    for sentence in sentences:
        # 1. 문장을 모델이 이해할 수 있는 숫자(토큰)로 변환
        # (출처: Hugging Face 라이브러리의 표준적인 토큰화 방식)
        inputs = tokenizer(
            sentence, 
            return_tensors="pt", # 결과를 PyTorch 텐서로 받음
            padding=True, 
            truncation=True, 
            max_length=512
        )

        # 2. 모델에게 예측 시키기
        # torch.no_grad()는 예측 시 불필요한 계산을 막아 속도를 높여주는 역할
        with torch.no_grad():
            logits = model(**inputs).logits

        # 3. 모델의 예측 결과(logits)를 실제 확률(0~1 사이)로 변환
        # softmax 함수는 모델의 출력값을 모든 클래스에 대한 확률의 합이 1이 되도록 변환합니다.
        probabilities = torch.nn.functional.softmax(logits, dim=-1)[0]
        
        # 4. '보이스피싱'일 확률(인덱스 1)을 가져옴
        # 우리는 학습 때 '일반'=0, '보이스피싱'=1 로 라벨링했기 때문에 1번 인덱스가 보이스피싱 확률입니다.
        phishing_prob = probabilities[1].item() # .item()은 텐서에서 숫자 값만 추출

        # 5. 확률이 50%를 넘으면 '보이스피싱', 아니면 '일반 대화'로 최종 판단
        if phishing_prob > 0.5:
            predicted_label = "보이스피싱"
        else:
            predicted_label = "일반 대화"
            
        # 6. 결과 저장
        results.append({
            "문장": sentence,
            "판단": predicted_label,
            "확률": phishing_prob * 100  # 확률을 퍼센트로 보기 좋게 변환
        })
        
    return results

In [14]:
   # --- 테스트에 사용할 문장들 정의 ---
    # 두 모델의 성능을 공정하게 비교하기 위해 동일한 문장을 사용합니다.
test_sentences = [
        "안녕하세요 고객님, 오늘 날씨가 정말 좋네요. 점심 식사는 하셨나요?", # 확실한 일반 대화
        "OO캐피탈인데요, 대출 심사 결과가 나왔습니다. 수수료 30만원을 먼저 입금해주셔야 합니다.", # 확실한 보이스피싱
        "고객님 본인 확인을 위해 성함과 생년월일을 말씀해주세요." # 약간 애매한 문장
    ]

In [15]:
 # --- 1. 학습 전 원본 모델 성능 테스트 ---
print("--- 1. 학습 전 원본 모델 성능 테스트 ---")
    
# 학습시키기 전의 순수 KoELECTRA 모델 이름
original_model_name = "monologg/koelectra-base-v3-discriminator"
# 원본 모델과 토크나이저 불러오기
# (출처: Hugging Face 라이브러리에서 사전 학습된 모델을 불러오는 표준 방식)
original_tokenizer = AutoTokenizer.from_pretrained(original_model_name)
original_model = AutoModelForSequenceClassification.from_pretrained(original_model_name, num_labels=2)

# 예측 함수 호출
original_predictions = predict_phishing(original_model, original_tokenizer, test_sentences)

# 결과 출력
for pred in original_predictions:
    print(f"문장: {pred['문장']}")
    print(f" -> 판단: {pred['판단']} (보이스피싱 확률: {pred['확률']:.2f}%)")
# --- 2. 파인튜닝 후 모델 성능 테스트 ---
print("\n--- 2. 파인튜닝 후 모델 성능 테스트 ---")

# 우리가 학습시켜서 저장해둔 모델의 폴더 경로
fine_tuned_model_path = "./fine-tuned-phishing-model"
# 파인튜닝된 모델과 토크나이저 불러오기
tuned_tokenizer = AutoTokenizer.from_pretrained(fine_tuned_model_path)
tuned_model = AutoModelForSequenceClassification.from_pretrained(fine_tuned_model_path)

# 예측 함수 호출
tuned_predictions = predict_phishing(tuned_model, tuned_tokenizer, test_sentences)

# 결과 출력
for pred in tuned_predictions:
    print(f"문장: {pred['문장']}")
    print(f" -> 판단: {pred['판단']} (보이스피싱 확률: {pred['확률']:.2f}%)")

--- 1. 학습 전 원본 모델 성능 테스트 ---


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


문장: 안녕하세요 고객님, 오늘 날씨가 정말 좋네요. 점심 식사는 하셨나요?
 -> 판단: 일반 대화 (보이스피싱 확률: 49.86%)
문장: OO캐피탈인데요, 대출 심사 결과가 나왔습니다. 수수료 30만원을 먼저 입금해주셔야 합니다.
 -> 판단: 보이스피싱 (보이스피싱 확률: 50.27%)
문장: 고객님 본인 확인을 위해 성함과 생년월일을 말씀해주세요.
 -> 판단: 일반 대화 (보이스피싱 확률: 49.20%)

--- 2. 파인튜닝 후 모델 성능 테스트 ---
문장: 안녕하세요 고객님, 오늘 날씨가 정말 좋네요. 점심 식사는 하셨나요?
 -> 판단: 보이스피싱 (보이스피싱 확률: 74.74%)
문장: OO캐피탈인데요, 대출 심사 결과가 나왔습니다. 수수료 30만원을 먼저 입금해주셔야 합니다.
 -> 판단: 보이스피싱 (보이스피싱 확률: 99.98%)
문장: 고객님 본인 확인을 위해 성함과 생년월일을 말씀해주세요.
 -> 판단: 보이스피싱 (보이스피싱 확률: 98.15%)
